# SportsMOT test split — part 2 of 2 → Kaggle staging (CPU, no GPU quota)

Downloads the **odd-indexed sequences** (sorted order) of the `test/`
split of [`Lekim89/sportsmot`](https://huggingface.co/datasets/Lekim89/sportsmot)
**inside Kaggle's network** and writes `sportsmot-test-2.tar` to this
kernel's output. Its sibling `prepare-data-1` stages the other,
**mutually exclusive** half — the full split (~95k files, ~21 GB) exceeds
Kaggle's 20 GB kernel-output cap, so it is staged as two disjoint halves.
`testing/index.kaggle.ipynb` mounts BOTH outputs and benchmarks the whole
split. The test split has no ground truth (withheld for the SportsMOT
challenge server), so the testing stage measures performance only.

Same downloader design as the other prepare-data siblings: fresh connection
per request, hard timeouts, per-file retries; on deadline the partial result
is saved as `sportsmot-test-2.partial.tar` and a re-push resumes from it.
The complete tar is only written when every selected file arrived.


In [ ]:
# --- 1. Config + fresh-connection downloader --------------------------------
import shutil
import subprocess
import tarfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

HF_DATASET = "Lekim89/sportsmot"
STAGE = Path("/tmp/stage")       # tree on the ROOT disk: /kaggle/working is a
                                 # ~20 GB volume, and tree + tar together for a
                                 # 12 GB half blew it once (ENOSPC mid-tar).
                                 # Only the output tar belongs in working.
TAR_OK = Path("/kaggle/working/sportsmot-test-2.tar")          # complete only
TAR_PART = Path("/kaggle/working/sportsmot-test-2.partial.tar")  # deadline hit
TIMEOUT = (10, 30)               # (connect, read): wedged sockets raise, never hang
WORKERS = 12
DEADLINE_S = 10 * 3600           # leave margin inside Kaggle's ~12h CPU session
MIN_SEQUENCES, MIN_JPGS = 25, 20_000  # per-half floors
START = time.time()

# Optional HF token from the external-secrets dataset -> higher rate limits.
def _read_hf_token():
    base = Path("/kaggle/input")
    hits = sorted(base.rglob("secrets")) if base.is_dir() else []
    for hit in hits:
        for line in hit.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None

_tok = _read_hf_token()
HEADERS = {"authorization": f"Bearer {_tok}"} if _tok else {}
print("HF auth:", "token found" if _tok else "none (lower rate limits)")


def list_test_files():
    """[(path, size), ...] for every file under test/, via the paginated tree API."""
    files = []
    url = f"https://huggingface.co/api/datasets/{HF_DATASET}/tree/main/test?recursive=1"
    while url:
        for attempt in range(3):
            try:
                r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
                r.raise_for_status()
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(5)
        files += [(e["path"], e.get("size", 0)) for e in r.json() if e.get("type") == "file"]
        url = r.links.get("next", {}).get("url")
    return files


def download_all(files, dest_root, workers=WORKERS):
    """Fresh connection per request; size-verified resume; returns (failed, skipped)."""
    req_headers = {**HEADERS, "Connection": "close"}
    lock = threading.Lock()
    state = {"done": 0, "t0": time.time()}
    failed, skipped = [], []

    def fetch(item):
        path, size = item
        if time.time() - START > DEADLINE_S:
            skipped.append(path)                  # out of session budget: save partial
            return
        dest = dest_root / path
        if dest.is_file() and (size == 0 or dest.stat().st_size == size):
            return                                # resume: already complete
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://huggingface.co/datasets/{HF_DATASET}/resolve/main/{path}"
        for attempt in range(4):
            try:
                r = requests.get(url, headers=req_headers, timeout=TIMEOUT)
                r.raise_for_status()
                tmp = dest.with_suffix(dest.suffix + ".part")
                tmp.write_bytes(r.content)
                tmp.rename(dest)
                return
            except Exception as e:
                if attempt == 3:
                    failed.append(path)
                    status = getattr(getattr(e, "response", None), "status_code", "")
                    if len(failed) <= 20:      # don't flood the log (2700+ once)
                        print(f"FAILED after 4 tries: {path} "
                              f"({type(e).__name__} {status})", flush=True)
                    elif len(failed) == 21:
                        print("(further failures suppressed — final count below)", flush=True)
                    return
                time.sleep(2 * (attempt + 1))

    def fetch_and_count(item):
        fetch(item)
        with lock:
            state["done"] += 1
            if state["done"] % 2000 == 0:
                rate = state["done"] / max(1, time.time() - state["t0"])
                print(f"  {state['done']}/{len(files)} files ({rate:.1f}/s)", flush=True)

    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(fetch_and_count, files))
    return failed, skipped


In [ ]:
# --- 2. Resume from a previous version's tar, if one is mounted --------------
# When this kernel's previous output (complete or partial) is attached as a
# kernel_source, extract it first so only the remainder is downloaded.
base = Path("/kaggle/input")
_prev = None
if base.is_dir():
    for pat in ("*/sportsmot-test-2*.tar", "*/*/sportsmot-test-2*.tar", "*/*/*/sportsmot-test-2*.tar"):
        hits = sorted(base.glob(pat))
        if hits:
            _prev = hits[0]
            break
if _prev is not None and not STAGE.exists():
    print(f"Resuming from previous output: {_prev}")
    STAGE.mkdir(parents=True)
    with tarfile.open(_prev) as tf:
        tf.extractall(STAGE)
    print("  extracted", sum(1 for _ in STAGE.rglob("*.jpg")), "frames")
else:
    print("No previous tar mounted — starting fresh.")


In [ ]:
# --- 3. Download -------------------------------------------------------------
STAGE.mkdir(parents=True, exist_ok=True)
files = list_test_files()
# PART 2 OF 2: the full test split (~95k files, ~21 GB) exceeds Kaggle's
# 20 GB kernel-output cap, so it is staged as two MUTUALLY EXCLUSIVE halves:
# this kernel takes the odd-indexed sequences of the sorted list; its
# sibling prepare-data-1 takes the rest. testing/index mounts both.
_seqs = sorted({f[0].split("/")[1] for f in files})
_keep = set(_seqs[1::2])
files = [f for f in files if f[0].split("/")[1] in _keep]
print(f"part 2: {len(_keep)}/{len(_seqs)} sequences selected")
total_gb = sum(s for _, s in files) / 1e9
print(f"{len(files)} files, {total_gb:.2f} GB to mirror")

t0 = time.time()
failed, skipped = download_all(files, STAGE)
print(f"download pass done in {(time.time() - t0) / 60:.1f} min — "
      f"{len(failed)} failed, {len(skipped)} skipped (deadline)")

# One serial retry sweep for stragglers (fresh connections, no concurrency).
if failed and not skipped:
    print("retrying failures serially...")
    still_failed, _ = download_all([f for f in files if f[0] in set(failed)], STAGE, workers=1)
    failed = still_failed

complete = not failed and not skipped
seqs = sorted(p for p in (STAGE / "test").iterdir() if p.is_dir())

n_jpg = sum(1 for _ in STAGE.glob("test/*/img1/*.jpg"))
print(f"staged: {len(seqs)} sequences, {n_jpg} frames (test split: no gt)")
if complete and (len(seqs) < MIN_SEQUENCES or n_jpg < MIN_JPGS):
    raise RuntimeError("counts below sanity floor — dataset layout changed upstream?")


In [ ]:
# --- 4. Tar the result as this kernel's output -------------------------------
# Complete -> sportsmot-test-2.tar (the name trainers trust).
# Deadline/failures -> sportsmot-test-2.partial.tar (next run resumes from it).
target = TAR_OK if complete else TAR_PART
for old in (TAR_OK, TAR_PART):
    if old.exists():
        old.unlink()
print(f"writing {target.name} ...")
subprocess.run(["tar", "-cf", str(target), "-C", str(STAGE), "test"], check=True)
shutil.rmtree(STAGE)          # keep the kernel output to just the tar
size_gb = target.stat().st_size / 1e9
print("=" * 62)
if complete:
    print(f"DATA STAGING COMPLETE — {target.name} ({size_gb:.2f} GB, {n_jpg} frames)")
    print("Evaluation kernels mount this kernel's output via kernel_sources.")
else:
    print(f"PARTIAL ({size_gb:.2f} GB, {n_jpg} frames) — commit this kernel again")
    print("with this version's output attached to resume the remainder.")
print("=" * 62)
